In [ ]:
import torch
from snntorch import functional as SF

from thesis_code.datasets import get_SHD_dataloader
from thesis_code.networks import train, test, models
from thesis_code.plotting import plot_performance

In [ ]:
dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

In [ ]:
SHD_trainloader, input_dim, classes = get_SHD_dataloader(
    batch_size=128,
    time_window=10000,
    train=True,
    shuffle=True,
    re_download=False,
)

SHD_testloader, _, _ = get_SHD_dataloader(
    batch_size=128,
    time_window=10000,
    train=False,
    shuffle=True,
    re_download=False,
)

net = models.build_basic_pv_som_srnn(
    n_in=input_dim,
    ns_hidden=[1000],
    n_out=len(classes),
    beta=0.9,
    percent_pv=20,
    percent_som=20,
).to(device)

In [ ]:
net, loss_hist, acc_hist = train(
    net=net,
    device=device,
    trainloader=SHD_trainloader,
    optimizer=torch.optim.Adam(net.parameters(), lr=2e-2, betas=(0.9, 0.999)),
    loss_fn=SF.mse_membrane_loss(on_target=1.05, off_target=0.2, time_var_targets=False),
    n_epochs=1,
    max_iters=None,
)
plot_performance(loss_hist, acc_hist)

In [ ]:
accuracy = test(
    net=net,
    device=device,
    testloader=SHD_testloader,
    max_iters=None,
)
print(f"Test Accuracy is: {accuracy}")